In [2]:

import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import webbrowser

# ============================================================
# TASK 6: DUAL-AXIS CHART — FREE VS PAID APPS
# Dashboard available only from 1 PM to 2 PM IST
# ============================================================

# 1. Load dataset
DATA_FILE = Path("Play Store Data.csv")

if not DATA_FILE.exists():
    raise FileNotFoundError(
        "Play Store Data.csv was not found. Keep it in the same project folder."
    )

df = pd.read_csv(DATA_FILE)
print("Dataset loaded:", df.shape)

# 2. Clean values
def clean_numeric(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("+", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

def size_to_mb(value):
    if pd.isna(value):
        return float("nan")

    value = str(value).strip().upper()

    if value in ("", "NAN", "VARIES WITH DEVICE"):
        return float("nan")

    try:
        if value.endswith("M"):
            return float(value[:-1])
        if value.endswith("K"):
            return float(value[:-1]) / 1024
        if value.endswith("G"):
            return float(value[:-1]) * 1024
        return float(value)
    except ValueError:
        return float("nan")

def android_version(value):
    if pd.isna(value):
        return float("nan")

    value = str(value).strip()
    if value.upper() in ("", "NAN", "VARIES WITH DEVICE"):
        return float("nan")

    # Read the first numeric version, e.g. "4.1 and up" -> 4.1
    import re
    match = re.search(r"\d+(?:\.\d+)?", value)
    return float(match.group()) if match else float("nan")

df["Installs"] = clean_numeric(df["Installs"])
df["Price"] = (
    df["Price"].astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df["Reviews"] = clean_numeric(df["Reviews"])
df["Size_MB"] = df["Size"].apply(size_to_mb)
df["Android_Version"] = df["Android Ver"].apply(android_version)

# Normalize Type and Content Rating
df["Type"] = df["Type"].astype(str).str.strip().str.title()
df["Content Rating"] = df["Content Rating"].astype(str).str.strip()

# 3. Apply required app-level filters
filtered = df[
    (df["Installs"] >= 10000) &
    (df["Price"].notna()) &
    (df["Price"] * df["Installs"] >= 10000) &
    (df["Android_Version"] > 4.0) &
    (df["Size_MB"] > 15) &
    (df["Content Rating"].str.casefold() == "everyone") &
    (df["App"].astype(str).str.len() <= 30) &
    (df["Category"].notna()) &
    (df["Type"].isin(["Free", "Paid"]))
].copy()

print("Rows after filters:", len(filtered))

# 4. Estimate revenue
# The dataset does not include actual revenue.
# This proxy is price multiplied by install count.
filtered["Estimated_Revenue"] = (
    filtered["Price"].fillna(0) * filtered["Installs"].fillna(0)
)

# 5. Identify the top 3 categories by installs after filtering
top_categories = (
    filtered.groupby("Category")["Installs"]
    .sum()
    .nlargest(3)
    .index.tolist()
)

filtered = filtered[filtered["Category"].isin(top_categories)].copy()

if filtered.empty or not top_categories:
    raise ValueError(
        "No data matched all filters. Review the filters or source dataset."
    )

# 6. Summarize Free vs Paid apps by category
summary = (
    filtered.groupby(["Category", "Type"])
    .agg(
        Average_Installs=("Installs", "mean"),
        Estimated_Revenue=("Estimated_Revenue", "sum"),
        App_Count=("App", "count")
    )
    .reset_index()
)

# Keep Free and Paid entries visible for every selected category.
categories = top_categories
types = ["Free", "Paid"]

print("Top 3 categories:", categories)
display(summary)

# 7. Create a dual-axis grouped chart
fig = go.Figure()

colors = {"Free": "#4C78A8", "Paid": "#F58518"}

for app_type in types:
    part = (
        summary[summary["Type"] == app_type]
        .set_index("Category")
        .reindex(categories)
    )

    fig.add_trace(
        go.Bar(
            x=categories,
            y=part["Average_Installs"].fillna(0),
            name=f"{app_type} — Average Installs",
            marker_color=colors[app_type],
            offsetgroup=app_type,
            yaxis="y",
            hovertemplate=(
                "Category: %{x}<br>"
                f"Type: {app_type}<br>"
                "Average Installs: %{y:,.0f}<extra></extra>"
            )
        )
    )

    fig.add_trace(
        go.Bar(
            x=categories,
            y=part["Estimated_Revenue"].fillna(0),
            name=f"{app_type} — Estimated Revenue",
            marker_color=colors[app_type],
            opacity=0.55,
            offsetgroup=app_type,
            yaxis="y2",
            hovertemplate=(
                "Category: %{x}<br>"
                f"Type: {app_type}<br>"
                "Estimated Revenue: $%{y:,.0f}<extra></extra>"
            )
        )
    )

fig.update_layout(
    title="Task 6: Average Installs and Estimated Revenue — Free vs Paid Apps",
    xaxis_title="Top 3 App Categories by Installs",
    yaxis=dict(title="Average Installs", rangemode="tozero"),
    yaxis2=dict(
        title="Estimated Revenue (USD)",
        overlaying="y",
        side="right",
        rangemode="tozero"
    ),
    barmode="group",
    template="plotly_white",
    height=700,
    hovermode="x unified",
    legend_title="Metric",
    margin=dict(l=80, r=100, t=100, b=100)
)

fig.show()

# 8. Generate time-restricted HTML dashboard
chart_html = fig.to_html(
    full_html=False,
    include_plotlyjs=True,
    config={"responsive": True, "displaylogo": False}
)

dashboard_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Task 6 - Free vs Paid App Analysis</title>
<style>
body {{
    font-family: Arial, sans-serif;
    margin: 0;
    padding: 20px;
    background: #f4f6f9;
    color: #222;
}}
.container {{
    max-width: 1400px;
    margin: auto;
    padding: 20px;
    background: white;
    border-radius: 12px;
     text-align: center;
}}
#notice {{
    text-align: center;
    padding: 40px 20px;
    font-size: 20px;
}}
.note {{
    color: #8a4b00;
    background: #fff4dc;
    padding: 12px;
    border-radius: 8px;
     text-align: center;
}}
</style>
</head>
<body>
<div class="container">
<h2> Google Play Store Analytics</h2>
<p class="note">
Revenue is estimated as app price × installs because the supplied dataset
does not contain actual revenue figures.
</p>
<div id="notice" style="display:none;">
    <h3>Dashboard currently unavailable</h3>
    <p>This dashboard is available only from 1:00 PM to before 2:00 PM IST.</p>
    <p id="currentTime"></p>
</div>
<div id="chart" style="display:none;">
{chart_html}
</div>
</div>
<script>
function checkDashboardTime() {{
    const parts = new Intl.DateTimeFormat("en-GB", {{
        timeZone: "Asia/Kolkata",
        hour: "2-digit",
        minute: "2-digit",
        hourCycle: "h23"
    }}).formatToParts(new Date());

    const hour = Number(parts.find(p => p.type === "hour").value);
    const minute = Number(parts.find(p => p.type === "minute").value);
    const currentMinutes = hour * 60 + minute;
    const allowed = currentMinutes >= 780 && currentMinutes < 840;

    document.getElementById("chart").style.display =
        allowed ? "block" : "none";
    document.getElementById("notice").style.display =
        allowed ? "none" : "block";

    document.getElementById("currentTime").textContent =
        "Current IST time: " +
        new Intl.DateTimeFormat("en-IN", {{
            timeZone: "Asia/Kolkata",
            hour: "2-digit",
            minute: "2-digit",
            second: "2-digit",
            hour12: true
        }}).format(new Date());
}}

checkDashboardTime();
setInterval(checkDashboardTime, 15000);
</script>
</body>
</html>
"""

OUTPUT_FILE = Path("dashboard_task6.html")
OUTPUT_FILE.write_text(dashboard_html, encoding="utf-8")

print("\nTask 6 completed.")
print("Dashboard saved at:", OUTPUT_FILE.resolve())
print("Display window: 1:00 PM to before 2:00 PM IST.")
print("Current IST:", datetime.now(ZoneInfo("Asia/Kolkata")).strftime("%I:%M:%S %p"))

webbrowser.open(OUTPUT_FILE.resolve().as_uri())
print("Dashboard opened in your default browser.")

Dataset loaded: (10841, 13)
Rows after filters: 33
Top 3 categories: ['PHOTOGRAPHY', 'FAMILY', 'PERSONALIZATION']


,Category,Type,Average_Installs,Estimated_Revenue,App_Count
0,FAMILY,Paid,381428.571429,5578300.0,7
1,PERSONALIZATION,Paid,670000.000000,1999900.0,3
2,PHOTOGRAPHY,Paid,1000000.000000,17970000.0,3



Task 6 completed.
Dashboard saved at: C:\Users\Ranjita\Downloads\Google-Play-Store-Analytics-main (1)\Google-Play-Store-Analytics-main\dashboard_task6.html
Display window: 1:00 PM to before 2:00 PM IST.
Current IST: 02:16:00 PM
Dashboard opened in your default browser.
